### Imports

In [0]:
from pyspark.sql.functions import (
    avg,
    sum,
    round,
    col
)

from framework.core.session import spark

from framework.core.configuration import (
    GOLD_LAYER
)

from framework.core.logger import (
    banner,
    success
)

from framework.io.delta import write_delta

### Banner

In [0]:
banner("Gold Layer - Executive Summary")

### Read Gold Tables

In [0]:
production = (

    spark.table(f"{GOLD_LAYER}.production_summary")

)

quality = (

    spark.table(f"{GOLD_LAYER}.quality_summary")

)

oee = (

    spark.table(f"{GOLD_LAYER}.oee_summary")

)

### Executive Dataset

In [0]:
# ============================================================================
# Executive Dataset
# ============================================================================

production = production.alias("p")
quality = quality.alias("q")

oee_daily = (

    oee

    .groupBy(

        "production_date"

    )

    .agg(

        round(

            avg("daily_oee"),

            2

        ).alias("plant_oee"),

        round(

            avg("availability_pct"),

            2

        ).alias("availability"),

        round(

            avg("performance_pct"),

            2

        ).alias("performance"),

        round(

            avg("quality_pct"),

            2

        ).alias("quality")

    )

)

executive = (

    production

    .join(

        quality,

        [

            "production_date",

            "planned_shift",

            "product_key"

        ],

        "left"

    )

    .join(

        oee_daily,

        "production_date",

        "left"

    )

    .select(

        col("p.production_date"),

        col("p.planned_shift"),

        col("p.product_key"),

        col("p.product_code"),

        col("p.product_name"),

        col("p.units_produced"),

        col("p.completed_work_orders"),

        col("q.tests_performed"),

        col("q.passed_tests"),

        col("q.failed_tests"),

        col("q.pass_rate"),

        col("plant_oee"),

        col("availability"),

        col("performance"),

        col("quality")

    )

)

### Final Select

In [0]:
executive = executive.select(

    "production_date",

    "planned_shift",

    "product_code",

    "product_name",

    "units_produced",

    "completed_work_orders",

    "tests_performed",

    "passed_tests",

    "failed_tests",

    "pass_rate",

    "plant_oee",

    "availability",

    "performance",

    "quality"

)

### Write Gold

In [0]:
write_delta(

    executive,

    f"{GOLD_LAYER}.executive_summary"

)

success("gold.executive_summary created successfully.")

### Validation

In [0]:
display(executive)

display(

    spark.sql(f"""

    SELECT

        ROUND(

            AVG(plant_oee),

            2

        ) plant_oee,

        SUM(units_produced) units,

        SUM(completed_work_orders) work_orders,

        ROUND(

            AVG(pass_rate),

            2

        ) pass_rate

    FROM {GOLD_LAYER}.executive_summary

    """)

)


